# SI-SDR Evaluation — All Three Pipelines (1D CNN, MFCC Ensemble, CNN14)

Combined, self-contained Colab notebook computing Scale-Invariant Signal-to-Distortion Ratio (SI-SDR)
for the U-Net traffic-noise denoiser in each of the paper's three pipelines.

Produces the exact numbers reported in **Table 8** of the manuscript.

**Structure:**
- Section 0 — shared setup (Drive mount, SI-SDR metric, combined results table at the end)
- Section 1 — Pipeline 1 (1D CNN, raw waveform, 22.05 kHz)
- Section 2 — Pipeline 2 (MFCC Ensemble, 44.1 kHz)
- Section 3 — Pipeline 3 (CNN14, log-mel, 32 kHz)
- Section 4 — combined Table 8 + save to `results/sisdr/`

Each pipeline section is fully independent (own U-Net class, own config, own checkpoint path) —
run them in any order, or just the one you need. Edit the `# <-- EDIT` paths in each section
before running.


## Section 0 — Shared setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DRIVE_BASE = Path("/content/drive/MyDrive/DSP Project")  # <-- EDIT if your project root differs


In [ ]:
def si_sdr(clean, denoised, eps=1e-8):
    """Scale-Invariant SDR in dB. clean, denoised: 1D numpy arrays, same length.
    Shared across all three pipelines below -- do not redefine per-section.
    """
    clean    = clean - np.mean(clean)
    denoised = denoised - np.mean(denoised)
    alpha  = np.dot(denoised, clean) / (np.dot(clean, clean) + eps)
    target = alpha * clean
    noise  = denoised - target
    return 10 * np.log10((np.sum(target**2) + eps) / (np.sum(noise**2) + eps))

# Collects results from all three sections below for the combined Table 8 at the end.
all_pipeline_results = {}


## Section 1 — Pipeline 1: Raw Waveform 1D CNN (22.05 kHz, base_ch=16)

Reproduces the SI-SDR rows for the 1D CNN pipeline (Table 8, `1D CNN` rows).

In [ ]:
# --- Pipeline 1 paths (EDIT to match your Drive setup) ----------------------
P1_DATA_PATH     = "/content/drive/MyDrive/Trial DSP/ESC-50-master"  # <-- EDIT
P1_AUDIO_PATH    = os.path.join(P1_DATA_PATH, "audio")
P1_META_PATH     = os.path.join(P1_DATA_PATH, "meta", "esc50.csv")
P1_UNET_CKPT     = os.path.join(P1_DATA_PATH, "unet_denoiser", "models", "unet_best.pth")  # <-- EDIT
P1_TRAFFIC_WAV   = os.path.join(P1_DATA_PATH, "traffic_noise.wav")  # <-- EDIT

# --- Pipeline 1 settings (must match training exactly) ----------------------
P1_SAMPLE_RATE  = 22050
P1_MAX_LEN      = P1_SAMPLE_RATE * 5
P1_TARGET_RMS   = 0.1
P1_TEST_FOLD    = 5
P1_N_FFT        = 512
P1_HOP_LENGTH   = 128
P1_WIN_LENGTH   = 512
P1_UNET_BASE_CH = 16
TRAFFIC_SNR_LEVELS = [-10, -5, 0, 5, 10]
SEED = 42


In [ ]:
# --- Pipeline 1: audio loading helpers --------------------------------------
def p1_rms_normalize(y, target_rms=P1_TARGET_RMS):
    rms = np.sqrt(np.mean(y ** 2))
    if rms < 1e-9:
        return y
    return (y * target_rms / rms).astype(np.float32)

def p1_load_wave(path):
    y, _ = librosa.load(path, sr=P1_SAMPLE_RATE, mono=True)
    if len(y) < P1_MAX_LEN:
        y = np.pad(y, (0, P1_MAX_LEN - len(y)))
    else:
        y = y[:P1_MAX_LEN]
    return p1_rms_normalize(y).astype(np.float32)

p1_meta = pd.read_csv(P1_META_PATH)
p1_test_meta = p1_meta[p1_meta["fold"] == P1_TEST_FOLD].reset_index(drop=True)
print(f"[P1] Test fold {P1_TEST_FOLD}: {len(p1_test_meta)} clips")

p1_clean_waves = []
for _, row in p1_test_meta.iterrows():
    path = os.path.join(P1_AUDIO_PATH, row["filename"])
    p1_clean_waves.append(p1_load_wave(path))
p1_clean_waves = np.array(p1_clean_waves, dtype=np.float32)
print(f"[P1] Loaded clean test waveforms: {p1_clean_waves.shape}")

if not os.path.exists(P1_TRAFFIC_WAV):
    print(f"[P1] Traffic noise not found at {P1_TRAFFIC_WAV} -- please upload it now...")
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    P1_TRAFFIC_WAV = f"/content/{fname}"
    Path(P1_TRAFFIC_WAV).write_bytes(uploaded[fname])

p1_traffic_audio, _ = librosa.load(P1_TRAFFIC_WAV, sr=P1_SAMPLE_RATE, mono=True)
print(f"[P1] Traffic noise loaded: {len(p1_traffic_audio)/P1_SAMPLE_RATE:.1f}s")

def p1_mix_traffic_noise(signal, traffic, snr_db):
    p_signal = np.mean(signal ** 2)
    p_noise  = np.mean(traffic ** 2) + 1e-12
    scale    = np.sqrt((p_signal / (10 ** (snr_db / 10))) / p_noise)
    return np.clip(signal + traffic * scale, -1.0, 1.0).astype(np.float32)

def p1_load_traffic_segment_cached(traffic_audio, n_samples, seed):
    y = traffic_audio
    if len(y) < n_samples:
        y = np.tile(y, int(np.ceil(n_samples / len(y))))
    rng   = np.random.default_rng(seed)
    start = rng.integers(0, len(y) - n_samples + 1)
    seg   = y[start: start + n_samples].astype(np.float32)
    rms   = np.sqrt(np.mean(seg ** 2))
    return (seg / (rms + 1e-9)).astype(np.float32)

def p1_add_traffic_noise_batch(X_clean, snr_db, seed):
    clip_seeds  = [seed * 100_000 + i for i in range(len(X_clean))]
    noisy_clips = [
        p1_mix_traffic_noise(
            X_clean[i],
            p1_load_traffic_segment_cached(p1_traffic_audio, P1_MAX_LEN, cs),
            snr_db,
        )
        for i, cs in enumerate(clip_seeds)
    ]
    return np.stack(noisy_clips, axis=0)


In [ ]:
# --- Pipeline 1: STFT/iSTFT + U-Net architecture (must match training) -----
_P1_WINDOW = torch.hann_window(P1_N_FFT, periodic=True)

def p1_stft_torch(wav):
    window = _P1_WINDOW.to(wav.device)
    return torch.stft(
        wav, n_fft=P1_N_FFT, hop_length=P1_HOP_LENGTH, win_length=P1_WIN_LENGTH,
        window=window, return_complex=True, center=True,
    )

def p1_istft_torch(stft_c, length):
    window = _P1_WINDOW.to(stft_c.device)
    return torch.istft(
        stft_c, n_fft=P1_N_FFT, hop_length=P1_HOP_LENGTH, win_length=P1_WIN_LENGTH,
        window=window, center=True, length=length,
    )

class P1_ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class P1_DenoiseUNet(nn.Module):
    def __init__(self, base_ch=P1_UNET_BASE_CH):
        super().__init__()
        c = base_ch
        self.enc1 = nn.Sequential(P1_ConvBNReLU(1,   c),   P1_ConvBNReLU(c,   c))
        self.enc2 = nn.Sequential(P1_ConvBNReLU(c,   c*2), P1_ConvBNReLU(c*2, c*2))
        self.enc3 = nn.Sequential(P1_ConvBNReLU(c*2, c*4), P1_ConvBNReLU(c*4, c*4))
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(P1_ConvBNReLU(c*4, c*8), P1_ConvBNReLU(c*8, c*8))
        self.up3  = nn.ConvTranspose2d(c*8, c*4, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(P1_ConvBNReLU(c*8, c*4), P1_ConvBNReLU(c*4, c*4))
        self.up2  = nn.ConvTranspose2d(c*4, c*2, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(P1_ConvBNReLU(c*4, c*2), P1_ConvBNReLU(c*2, c*2))
        self.up1  = nn.ConvTranspose2d(c*2, c,   kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(P1_ConvBNReLU(c*2, c),   P1_ConvBNReLU(c, c))
        self.out_conv = nn.Conv2d(c, 1, kernel_size=1)

    def forward(self, x):
        B, C, Fdim, Tdim = x.shape
        pad_f = (8 - Fdim % 8) % 8
        pad_t = (8 - Tdim % 8) % 8
        x_p = F.pad(x, (0, pad_t, 0, pad_f))
        e1 = self.enc1(x_p)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        mask = torch.sigmoid(self.out_conv(d1))
        return mask[:, :, :Fdim, :Tdim]

P1_EPS = 1e-8

def p1_mag_phase(wav):
    spec  = p1_stft_torch(wav)
    mag   = torch.abs(spec)
    phase = spec / (mag + P1_EPS)
    return mag, phase

def p1_denoise_wav_batch_torch(unet_model, noisy_wav):
    mag, phase = p1_mag_phase(noisy_wav)
    x_in  = torch.log1p(mag).unsqueeze(1)
    mask  = unet_model(x_in).squeeze(1)
    d_spec = (mask * mag) * phase
    return p1_istft_torch(d_spec, length=noisy_wav.shape[-1])

@torch.no_grad()
def p1_dsp_pipeline(X, unet, batch_size=32):
    out = []
    for start in range(0, len(X), batch_size):
        batch_np = X[start:start + batch_size]
        batch_t  = torch.tensor(batch_np, dtype=torch.float32, device=device)
        denoised_t = p1_denoise_wav_batch_torch(unet, batch_t)
        denoised_np = denoised_t.cpu().numpy()
        for d in denoised_np:
            out.append(p1_rms_normalize(d))
    return np.array(out, dtype=np.float32)


In [ ]:
# --- Pipeline 1: load checkpoint + run SI-SDR eval --------------------------
assert os.path.exists(P1_UNET_CKPT), (
    f"[P1] Checkpoint not found at {P1_UNET_CKPT}. Edit P1_UNET_CKPT above."
)

p1_unet = P1_DenoiseUNet(base_ch=P1_UNET_BASE_CH).to(device)
p1_unet.load_state_dict(torch.load(P1_UNET_CKPT, map_location=device))
p1_unet.eval()
print(f"[P1] Loaded U-Net checkpoint from {P1_UNET_CKPT}")
print(f"[P1] Parameters: {sum(p.numel() for p in p1_unet.parameters()):,}")

p1_results = {}
print(f"\n{'SNR':>6} | {'SI-SDR noisy':>13} | {'SI-SDR denoised':>16} | {'Improvement':>11}")
print("-" * 60)

for snr in TRAFFIC_SNR_LEVELS:
    X_noisy    = p1_add_traffic_noise_batch(p1_clean_waves, snr, seed=SEED)
    X_denoised = p1_dsp_pipeline(X_noisy, p1_unet)

    noisy_scores    = [si_sdr(p1_clean_waves[i], X_noisy[i])    for i in range(len(p1_clean_waves))]
    denoised_scores = [si_sdr(p1_clean_waves[i], X_denoised[i]) for i in range(len(p1_clean_waves))]

    n_mean, n_std = np.mean(noisy_scores), np.std(noisy_scores)
    d_mean, d_std = np.mean(denoised_scores), np.std(denoised_scores)
    improvement = d_mean - n_mean

    p1_results[snr] = {
        "si_sdr_noisy_mean": n_mean, "si_sdr_noisy_std": n_std,
        "si_sdr_denoised_mean": d_mean, "si_sdr_denoised_std": d_std,
        "improvement": improvement,
    }
    print(f"{snr:>5}dB | {n_mean:>7.2f} \u00b1 {n_std:<4.2f} | {d_mean:>10.2f} \u00b1 {d_std:<4.2f} | {improvement:>+10.2f} dB")

all_pipeline_results["1D CNN"] = p1_results

p1_df = pd.DataFrame(p1_results).T
p1_df.index.name = "SNR_dB"
p1_out_csv = os.path.join(P1_DATA_PATH, "sisdr_results_pipeline1.csv")
p1_df.to_csv(p1_out_csv)
print(f"\nSaved -> {p1_out_csv}")
p1_df


## Section 2 — Pipeline 2: MFCC Ensemble (44.1 kHz, U-Net base_ch=32)

SI-SDR only needs clean vs. denoised **waveforms** — the MFCC ensemble classifier itself is not
needed here, only the U-Net. Reproduces Table 8's `MFCC` rows.

In [ ]:
# --- Pipeline 2 paths (EDIT to match your Drive setup) ----------------------
P2_AUDIO_DIR   = DRIVE_BASE / "esc50_data/ESC-50-master/audio"
P2_META_PATH   = DRIVE_BASE / "esc50_data/ESC-50-master/meta/esc50.csv"

# One of your trained U-Net checkpoints, e.g.:
#   ".../mfcc_unet_denoiser_multirun/models/unet_run1_seed42_traffic-gaussian.pth"
P2_UNET_CKPT = str(DRIVE_BASE / "mfcc_unet_denoiser_multirun" / "models" /
                    "unet_run1_seed42_traffic-gaussian.pth")  # <-- EDIT filename if different

P2_TRAFFIC_WAV = str(DRIVE_BASE / "traffic_noise.wav")  # <-- EDIT if different

# --- Pipeline 2 settings (must match training exactly) ----------------------
P2_SR           = 44_100
P2_CLIP_LEN     = P2_SR * 5
P2_TARGET_RMS   = 0.1
P2_TEST_FOLD    = 5
P2_N_FFT_UNET   = 1024
P2_HOP_UNET     = 256
P2_WIN_UNET     = 1024
P2_UNET_BASE_CH = 32


In [ ]:
# --- Pipeline 2: audio loading helpers --------------------------------------
def p2_rms_normalize(y, target=P2_TARGET_RMS):
    rms = np.sqrt(np.mean(y ** 2))
    if rms < 1e-9:
        return y.astype(np.float32)
    return (y * (target / rms)).astype(np.float32)

def p2_pad_or_trim(y):
    if len(y) > P2_CLIP_LEN:
        return y[:P2_CLIP_LEN]
    return np.pad(y, (0, P2_CLIP_LEN - len(y))).astype(np.float32)

def p2_load_audio_clip(filepath):
    y, _ = librosa.load(filepath, sr=P2_SR, mono=True)
    y = p2_rms_normalize(p2_pad_or_trim(y))
    return y.astype(np.float32)

p2_meta_df = pd.read_csv(P2_META_PATH)
p2_test_meta = p2_meta_df[p2_meta_df["fold"] == P2_TEST_FOLD].reset_index(drop=True)
print(f"[P2] Test fold {P2_TEST_FOLD}: {len(p2_test_meta)} clips")

p2_clean_waves = []
for _, row in p2_test_meta.iterrows():
    fpath = str(P2_AUDIO_DIR / row["filename"])
    p2_clean_waves.append(p2_load_audio_clip(fpath))
p2_clean_waves = np.array(p2_clean_waves, dtype=np.float32)
print(f"[P2] Loaded clean test waveforms: {p2_clean_waves.shape}")

if not os.path.exists(P2_TRAFFIC_WAV):
    print(f"[P2] Traffic noise not found at {P2_TRAFFIC_WAV} -- please upload it now...")
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    P2_TRAFFIC_WAV = f"/content/{fname}"
    Path(P2_TRAFFIC_WAV).write_bytes(uploaded[fname])

p2_raw_noise, _ = librosa.load(P2_TRAFFIC_WAV, sr=P2_SR, mono=True)
p2_raw_noise = p2_raw_noise.astype(np.float32)
print(f"[P2] Traffic noise loaded: {len(p2_raw_noise)/P2_SR:.1f}s")

def p2_fit_noise_to_length(noise, length, rng):
    if len(noise) < length:
        reps = int(np.ceil(length / len(noise)))
        noise = np.tile(noise, reps)
    if len(noise) > length:
        max_start = len(noise) - length
        start = int(rng.integers(0, max_start + 1))
        noise = noise[start:start + length]
    return noise

def p2_add_traffic_noise(wav_np, target_snr_db, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    noise = p2_fit_noise_to_length(p2_raw_noise, len(wav_np), rng)
    p_sig = np.mean(wav_np ** 2)
    p_n   = p_sig / (10 ** (target_snr_db / 10.0))
    noise = noise * np.sqrt(p_n / (np.mean(noise ** 2) + 1e-10))
    return np.clip(wav_np + noise, -1.0, 1.0).astype(np.float32)


In [ ]:
# --- Pipeline 2: U-Net architecture (must match training exactly) ----------
class P2_DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class P2_UNetDenoiser(nn.Module):
    def __init__(self, base_ch: int = 32):
        super().__init__()
        self.enc1 = P2_DoubleConv(1,         base_ch)
        self.enc2 = P2_DoubleConv(base_ch,   base_ch * 2)
        self.enc3 = P2_DoubleConv(base_ch*2, base_ch * 4)
        self.pool = nn.MaxPool2d(2)
        self.bot  = P2_DoubleConv(base_ch*4, base_ch * 8)
        self.up3  = nn.ConvTranspose2d(base_ch*8, base_ch*4, 2, stride=2)
        self.dec3 = P2_DoubleConv(base_ch*8, base_ch*4)
        self.up2  = nn.ConvTranspose2d(base_ch*4, base_ch*2, 2, stride=2)
        self.dec2 = P2_DoubleConv(base_ch*4, base_ch*2)
        self.up1  = nn.ConvTranspose2d(base_ch*2, base_ch,   2, stride=2)
        self.dec1 = P2_DoubleConv(base_ch*2, base_ch)
        self.head = nn.Sequential(nn.Conv2d(base_ch, 1, 1), nn.Sigmoid())

    def _pad_to_match(self, x, ref):
        dh = ref.shape[2] - x.shape[2]
        dw = ref.shape[3] - x.shape[3]
        if dh != 0 or dw != 0:
            x = F.pad(x, [0, dw, 0, dh])
        return x

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bot(self.pool(e3))
        d3 = self.dec3(torch.cat([self._pad_to_match(self.up3(b),  e3), e3], dim=1))
        d2 = self.dec2(torch.cat([self._pad_to_match(self.up2(d3), e2), e2], dim=1))
        d1 = self.dec1(torch.cat([self._pad_to_match(self.up1(d2), e1), e1], dim=1))
        return self.head(d1)

def p2_unet_denoise_batch(unet, wav_batch_np):
    unet.eval()
    with torch.no_grad():
        window = torch.hann_window(P2_WIN_UNET)
        t = torch.tensor(wav_batch_np, dtype=torch.float32)
        S = torch.stft(
            t, n_fft=P2_N_FFT_UNET, hop_length=P2_HOP_UNET,
            win_length=P2_WIN_UNET, window=window,
            return_complex=True
        )
        mag      = S.abs().unsqueeze(1).to(device)
        max_val  = mag.amax(dim=(2, 3), keepdim=True).clamp(min=1e-8)
        mag_norm = mag / max_val
        mask     = unet(mag_norm)
        masked_mag = (mask * mag_norm) * max_val
        masked_mag = masked_mag.squeeze(1).cpu()
        phase      = torch.angle(S)
        S_hat      = masked_mag * torch.exp(1j * phase)
        wav_out    = torch.istft(
            S_hat, n_fft=P2_N_FFT_UNET, hop_length=P2_HOP_UNET,
            win_length=P2_WIN_UNET, window=window,
            length=wav_batch_np.shape[1],
        )
    return wav_out.numpy().astype(np.float32)


In [ ]:
# --- Pipeline 2: load checkpoint + run SI-SDR eval --------------------------
assert os.path.exists(P2_UNET_CKPT), (
    f"[P2] Checkpoint not found at {P2_UNET_CKPT}. Edit P2_UNET_CKPT above -- "
    f"check the exact filename in your mfcc_unet_denoiser_multirun/models/ folder."
)

p2_unet = P2_UNetDenoiser(base_ch=P2_UNET_BASE_CH).to(device)
p2_unet.load_state_dict(torch.load(P2_UNET_CKPT, map_location=device))
p2_unet.eval()
print(f"[P2] Loaded U-Net checkpoint from {P2_UNET_CKPT}")
print(f"[P2] Parameters: {sum(p.numel() for p in p2_unet.parameters()):,}")

P2_BATCH_SIZE = 16
p2_results = {}
print(f"\n{'SNR':>6} | {'SI-SDR noisy':>13} | {'SI-SDR denoised':>16} | {'Improvement':>11}")
print("-" * 60)

for snr in TRAFFIC_SNR_LEVELS:
    rng = np.random.default_rng(SEED)
    X_noisy = np.stack([p2_add_traffic_noise(c, snr, rng=rng) for c in p2_clean_waves])

    X_denoised = []
    for start in range(0, len(X_noisy), P2_BATCH_SIZE):
        batch = X_noisy[start:start + P2_BATCH_SIZE]
        X_denoised.append(p2_unet_denoise_batch(p2_unet, batch))
    X_denoised = np.concatenate(X_denoised, axis=0)

    noisy_scores    = [si_sdr(p2_clean_waves[i], X_noisy[i])    for i in range(len(p2_clean_waves))]
    denoised_scores = [si_sdr(p2_clean_waves[i], X_denoised[i]) for i in range(len(p2_clean_waves))]

    n_mean, n_std = np.mean(noisy_scores), np.std(noisy_scores)
    d_mean, d_std = np.mean(denoised_scores), np.std(denoised_scores)
    improvement = d_mean - n_mean

    p2_results[snr] = {
        "si_sdr_noisy_mean": n_mean, "si_sdr_noisy_std": n_std,
        "si_sdr_denoised_mean": d_mean, "si_sdr_denoised_std": d_std,
        "improvement": improvement,
    }
    print(f"{snr:>5}dB | {n_mean:>7.2f} \u00b1 {n_std:<4.2f} | {d_mean:>10.2f} \u00b1 {d_std:<4.2f} | {improvement:>+10.2f} dB")

all_pipeline_results["MFCC"] = p2_results

p2_df = pd.DataFrame(p2_results).T
p2_df.index.name = "SNR_dB"
p2_out_csv = str(DRIVE_BASE / "sisdr_results_pipeline2_mfcc.csv")
p2_df.to_csv(p2_out_csv)
print(f"\nSaved -> {p2_out_csv}")
p2_df


## Section 3 — Pipeline 3: CNN14 (32 kHz, U-Net base_ch=16, FFT=1024/hop=320)

Reproduces Table 8's `CNN14` rows. Note this pipeline's U-Net uses a **different** FFT/hop
(1024/320) and sample rate (32 kHz) than Pipeline 1, matching the CNN14 notebook's log-mel
front end -- do not reuse Pipeline 1's config here.

In [ ]:
# --- Pipeline 3 paths (EDIT to match your Drive setup) ----------------------
P3_DATA_PATH  = "/content/drive/MyDrive/DSP Project/esc50_data/ESC-50-master"  # <-- EDIT
P3_AUDIO_DIR  = Path(P3_DATA_PATH) / "audio"
P3_META_PATH  = Path(P3_DATA_PATH) / "meta" / "esc50.csv"

P3_UNET_CKPT_PATH = Path(
    "/content/drive/MyDrive/DSP Project/traffic_denoiser_unet/models/unet_best.pth"
)  # <-- EDIT if different

P3_TRAFFIC_WAV = "/content/drive/MyDrive/DSP Project/traffic_noise.wav"  # <-- EDIT if different

# --- Pipeline 3 settings (must match the CNN14 notebook exactly) -----------
P3_SR_MODEL     = 32_000
P3_CLIP_SAMPLES = 160_000
P3_N_FFT        = 1024
P3_HOP_LENGTH   = 320
P3_WIN_LENGTH   = 1024
P3_UNET_BASE_CH = 16
P3_OUTER_TEST_FOLD = 4


In [ ]:
# --- Pipeline 3: audio loading utilities ------------------------------------
import torchaudio
import torchaudio.transforms as T

def p3_load_and_resample(path: str) -> torch.Tensor:
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != P3_SR_MODEL:
        waveform = T.Resample(orig_freq=sr, new_freq=P3_SR_MODEL)(waveform)
    return waveform

def p3_fix_clip_length(waveform: torch.Tensor) -> torch.Tensor:
    n      = waveform.shape[-1]
    target = P3_CLIP_SAMPLES
    if n < target:
        waveform = torch.nn.functional.pad(waveform, (0, target - n))
    elif n > target:
        start    = (n - target) // 2
        waveform = waveform[:, start: start + target]
    return waveform

def p3_load_wav_np(file_path):
    wav = p3_fix_clip_length(p3_load_and_resample(file_path))
    return wav.numpy()  # (1, L)

if not os.path.exists(P3_TRAFFIC_WAV):
    print(f"[P3] Traffic noise not found at {P3_TRAFFIC_WAV} -- please upload it now...")
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    fname = list(uploaded.keys())[0]
    P3_TRAFFIC_WAV = f"/content/{fname}"
    Path(P3_TRAFFIC_WAV).write_bytes(uploaded[fname])

_p3_real_noise_wav, _p3_sr_in = torchaudio.load(P3_TRAFFIC_WAV)
if _p3_sr_in != P3_SR_MODEL:
    _p3_real_noise_wav = torchaudio.functional.resample(_p3_real_noise_wav, _p3_sr_in, P3_SR_MODEL)
if _p3_real_noise_wav.shape[0] > 1:
    _p3_real_noise_wav = _p3_real_noise_wav.mean(dim=0, keepdim=True)
p3_raw_noise = _p3_real_noise_wav.squeeze(0).numpy().astype(np.float32)
print(f"[P3] Traffic noise loaded: {len(p3_raw_noise)/P3_SR_MODEL:.1f}s")

def p3_fit_noise_to_length(noise, length, rng):
    if len(noise) < length:
        reps  = int(np.ceil(length / len(noise)))
        noise = np.tile(noise, reps)
    if len(noise) > length:
        max_start = len(noise) - length
        start     = int(rng.integers(0, max_start + 1))
        noise     = noise[start:start + length]
    return noise[:length]

def p3_add_real_noise(wav_np, target_snr_db, noise_clip=None, rng=None):
    if noise_clip is None:
        noise_clip = p3_raw_noise
    if rng is None:
        rng = np.random.default_rng()
    sig   = wav_np[0]
    noise = p3_fit_noise_to_length(noise_clip, len(sig), rng)
    p_sig = np.mean(sig ** 2)
    p_n   = p_sig / (10 ** (target_snr_db / 10.0))
    noise = noise * np.sqrt(p_n / (np.mean(noise ** 2) + 1e-10))
    return np.clip(sig + noise, -1.0, 1.0).astype(np.float32)[np.newaxis, :]


In [ ]:
# --- Pipeline 3: STFT/iSTFT + U-Net architecture (must match training) -----
_P3_WINDOW = torch.hann_window(P3_N_FFT, periodic=True)

def p3_stft_torch(wav, dev=None):
    window = _P3_WINDOW.to(wav.device if dev is None else dev)
    return torch.stft(
        wav, n_fft=P3_N_FFT, hop_length=P3_HOP_LENGTH, win_length=P3_WIN_LENGTH,
        window=window, return_complex=True, center=True,
    )

def p3_istft_torch(stft_c, length, dev=None):
    window = _P3_WINDOW.to(stft_c.device if dev is None else dev)
    return torch.istft(
        stft_c, n_fft=P3_N_FFT, hop_length=P3_HOP_LENGTH, win_length=P3_WIN_LENGTH,
        window=window, center=True, length=length,
    )

class P3_ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class P3_DenoiseUNet(nn.Module):
    def __init__(self, base_ch=16):
        super().__init__()
        c = base_ch
        self.enc1 = nn.Sequential(P3_ConvBNReLU(1,   c),   P3_ConvBNReLU(c,   c))
        self.enc2 = nn.Sequential(P3_ConvBNReLU(c,   c*2), P3_ConvBNReLU(c*2, c*2))
        self.enc3 = nn.Sequential(P3_ConvBNReLU(c*2, c*4), P3_ConvBNReLU(c*4, c*4))
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = nn.Sequential(P3_ConvBNReLU(c*4, c*8), P3_ConvBNReLU(c*8, c*8))
        self.up3  = nn.ConvTranspose2d(c*8, c*4, kernel_size=2, stride=2)
        self.dec3 = nn.Sequential(P3_ConvBNReLU(c*8, c*4), P3_ConvBNReLU(c*4, c*4))
        self.up2  = nn.ConvTranspose2d(c*4, c*2, kernel_size=2, stride=2)
        self.dec2 = nn.Sequential(P3_ConvBNReLU(c*4, c*2), P3_ConvBNReLU(c*2, c*2))
        self.up1  = nn.ConvTranspose2d(c*2, c,   kernel_size=2, stride=2)
        self.dec1 = nn.Sequential(P3_ConvBNReLU(c*2, c),   P3_ConvBNReLU(c, c))
        self.out_conv = nn.Conv2d(c, 1, kernel_size=1)

    def forward(self, x):
        B, C, Fdim, Tdim = x.shape
        pad_f = (8 - Fdim % 8) % 8
        pad_t = (8 - Tdim % 8) % 8
        x_p = F.pad(x, (0, pad_t, 0, pad_f))
        e1 = self.enc1(x_p)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        mask = torch.sigmoid(self.out_conv(d1))
        return mask[:, :, :Fdim, :Tdim]

P3_EPS = 1e-8

def p3_wav_to_mag_phase(wav):
    spec = p3_stft_torch(wav)
    mag  = torch.abs(spec)
    phase = spec / (mag + P3_EPS)
    return mag, phase

def p3_denoise_wav_batch(unet, noisy_wav):
    mag, phase = p3_wav_to_mag_phase(noisy_wav)
    x_in = torch.log1p(mag).unsqueeze(1)
    mask = unet(x_in).squeeze(1)
    denoised_mag = mask * mag
    denoised_spec = denoised_mag * phase
    return p3_istft_torch(denoised_spec, length=noisy_wav.shape[-1])

@torch.no_grad()
def p3_unet_denoise(unet, noisy_np):
    """noisy_np: (1, L) numpy -> denoised (1, L) numpy."""
    wav = torch.tensor(noisy_np, dtype=torch.float32, device=device)
    denoised = p3_denoise_wav_batch(unet, wav)
    return denoised.cpu().numpy().astype(np.float32)


In [ ]:
# --- Pipeline 3: load checkpoint, metadata, run SI-SDR eval -----------------
assert P3_UNET_CKPT_PATH.exists(), (
    f"[P3] Checkpoint not found at {P3_UNET_CKPT_PATH}. Edit P3_UNET_CKPT_PATH above."
)

p3_unet = P3_DenoiseUNet(base_ch=P3_UNET_BASE_CH).to(device)
p3_unet.load_state_dict(torch.load(P3_UNET_CKPT_PATH, map_location=device))
p3_unet.eval()
print(f"[P3] Loaded U-Net checkpoint from {P3_UNET_CKPT_PATH}")
print(f"[P3] Parameters: {sum(p.numel() for p in p3_unet.parameters()):,}")

p3_df_meta = pd.read_csv(P3_META_PATH)
p3_df_meta["file_path"] = p3_df_meta["filename"].apply(lambda f: str(P3_AUDIO_DIR / f))
p3_df_test = p3_df_meta[p3_df_meta["fold"] == P3_OUTER_TEST_FOLD].reset_index(drop=True)
print(f"[P3] Evaluating on {len(p3_df_test)} clips (fold {P3_OUTER_TEST_FOLD})")

p3_results_list = []
print(f"\n{'SNR':>6} | {'SI-SDR noisy':>13} | {'SI-SDR denoised':>16} | {'Improvement':>11}")
print("-" * 60)

for snr in TRAFFIC_SNR_LEVELS:
    noisy_sisdrs, denoised_sisdrs = [], []
    eval_rng = np.random.default_rng(SEED + int(snr))

    for _, row in p3_df_test.iterrows():
        clean_np    = p3_load_wav_np(row["file_path"])
        noisy_np    = p3_add_real_noise(clean_np, float(snr), rng=eval_rng)
        denoised_np = p3_unet_denoise(p3_unet, noisy_np)

        noisy_sisdrs.append(si_sdr(clean_np[0], noisy_np[0]))
        denoised_sisdrs.append(si_sdr(clean_np[0], denoised_np[0]))

    n_mean, n_std = np.mean(noisy_sisdrs), np.std(noisy_sisdrs)
    d_mean, d_std = np.mean(denoised_sisdrs), np.std(denoised_sisdrs)
    improvement = d_mean - n_mean

    print(f"{snr:>5}dB | {n_mean:>7.2f} \u00b1 {n_std:<4.2f} | {d_mean:>10.2f} \u00b1 {d_std:<4.2f} | {improvement:>+10.2f} dB")

    p3_results_list.append({
        "SNR_dB": snr,
        "si_sdr_noisy_mean": n_mean, "si_sdr_noisy_std": n_std,
        "si_sdr_denoised_mean": d_mean, "si_sdr_denoised_std": d_std,
        "improvement": improvement,
        "n_clips": len(p3_df_test),
    })

p3_df = pd.DataFrame(p3_results_list).set_index("SNR_dB")
all_pipeline_results["CNN14"] = {row["SNR_dB"] if False else snr: None for snr in []}  # placeholder, filled below
all_pipeline_results["CNN14"] = {r["SNR_dB"]: {k: v for k, v in r.items() if k not in ("SNR_dB", "n_clips")} for r in p3_results_list}

p3_out_csv = "/content/drive/MyDrive/DSP Project/traffic_denoiser_unet/results/sisdr_pipeline3_cnn14.csv"
os.makedirs(os.path.dirname(p3_out_csv), exist_ok=True)
p3_df.to_csv(p3_out_csv)
print(f"\nSaved -> {p3_out_csv}")
p3_df


## Section 4 — Combined Table 8 (all three pipelines)

Assembles the manuscript's Table 8 directly from the three sections above. Run Sections 1-3
first (in any order) so `all_pipeline_results` is fully populated.

In [ ]:
# --- Combine into a single manuscript-format table ---------------------------
rows = []
for pipeline_name in ["1D CNN", "MFCC", "CNN14"]:
    if pipeline_name not in all_pipeline_results:
        print(f"Skipping {pipeline_name} -- run its section above first.")
        continue
    for snr in TRAFFIC_SNR_LEVELS:
        r = all_pipeline_results[pipeline_name][snr]
        rows.append({
            "Pipeline": pipeline_name,
            "SNR": snr,
            "Noisy SI-SDR (dB)": f"{r['si_sdr_noisy_mean']:.2f} \u00b1 {r['si_sdr_noisy_std']:.2f}",
            "U-Net SI-SDR (dB)": f"{r['si_sdr_denoised_mean']:.2f} \u00b1 {r['si_sdr_denoised_std']:.2f}",
            "Improvement (dB)": f"+{r['improvement']:.2f}",
        })

table8_df = pd.DataFrame(rows)
print(table8_df.to_string(index=False))

table8_out = str(DRIVE_BASE / "table8_sisdr_combined.csv")
table8_df.to_csv(table8_out, index=False)
print(f"\nSaved combined Table 8 -> {table8_out}")
table8_df
